In [ ]:
from src.common.lake import SILVER_AGGTRADES, read_lake
from src.common.spark import get_spark

spark = get_spark("explore")
spark.sparkContext.setLogLevel("ERROR")

# Silver is Delta now. Reading the path with spark.read.parquet() would
# bypass the transaction log and pick up superseded files, so always go
# through read_lake().
df = read_lake(spark, SILVER_AGGTRADES)

df.printSchema()
df.show(5)
print(df.count())
df.groupBy("symbol").count().show()

In [3]:
df.groupBy("symbol", "file_month").count().orderBy(
    "symbol", "file_month").show(30)

+-------+----------+--------+
| symbol|file_month|   count|
+-------+----------+--------+
|BNBUSDT|   2026-06|10058771|
|BNBUSDT|   2026-07| 5772626|
|BNBUSDT|   2026-08| 6961577|
|BNBUSDT|   2026-09| 8686149|
|BNBUSDT|   2026-10| 8015121|
|BTCUSDT|   2026-05|21080265|
|BTCUSDT|   2026-06|34074359|
|BTCUSDT|   2026-07|24186730|
|BTCUSDT|   2026-08|23939440|
|BTCUSDT|   2026-09|26690426|
|BTCUSDT|   2026-10|25749901|
|ETHUSDT|   2026-06|30664042|
|ETHUSDT|   2026-07|21128759|
|ETHUSDT|   2026-08|20462172|
|ETHUSDT|   2026-09|22267530|
|ETHUSDT|   2026-10|18872673|
|SOLUSDT|   2026-06| 5116327|
|SOLUSDT|   2026-07| 3679293|
|SOLUSDT|   2026-08| 4977119|
|SOLUSDT|   2026-09| 5921716|
|SOLUSDT|   2026-10| 4992459|
|XRPUSDT|   2026-05| 3215247|
|XRPUSDT|   2026-06| 4178649|
|XRPUSDT|   2026-07| 2644553|
|XRPUSDT|   2026-08| 6016728|
|XRPUSDT|   2026-09|10004075|
|XRPUSDT|   2026-10| 4740009|
+-------+----------+--------+



In [4]:
from pyspark.sql import functions as F

btc = df.filter((F.col("symbol") == "BTCUSDT") &
                (F.col("file_month") == "2026-10"))
candles = btc.groupBy(F.date_trunc("hour", "trade_ts").alias("hour")).agg(
    F.min("price").alias("low"),
    F.max("price").alias("high"),
    F.sum("quantity").alias("volume"),
    F.count("*").alias("trades"),
)
candles.orderBy("hour").show(5)

+-------------------+--------------+--------------+------------+------+
|               hour|           low|          high|      volume|trades|
+-------------------+--------------+--------------+------------+------+
|2026-10-01 00:00:00|83429.80000000|83660.01000000|451.69518000| 20970|
|2026-10-01 01:00:00|83482.20000000|83692.00000000|343.44740000| 18869|
|2026-10-01 02:00:00|83410.39000000|83668.01000000|362.14695000| 18996|
|2026-10-01 03:00:00|83424.58000000|83780.14000000|325.01626000| 17564|
|2026-10-01 04:00:00|83720.40000000|84207.53000000|514.62822000| 27439|
+-------------------+--------------+--------------+------------+------+
only showing top 5 rows



In [ ]:
import os
from src.common.spark import get_spark

spark = get_spark("jdbc_test")

url = (
    f"jdbc:sqlserver://{os.environ['MSSQL_HOST']}:{os.environ['MSSQL_PORT']};"
    f"databaseName={os.environ['MSSQL_DB']};encrypt=true;trustServerCertificate=true"
)

dim_time = (
    spark.read.format("jdbc")
    .option("url", url)
    .option("dbtable", "gold.dim_time")
    .option("user", os.environ["MSSQL_USER"])
    .option("password", os.environ["MSSQL_PASSWORD"])
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)
dim_time.count()

1440

In [6]:
from pyspark.sql import functions as F
from src.transform.gold_candles import build_candles_1m

day = df.filter((F.col("symbol") == "BTCUSDT") &
                (F.col("trade_date") == "2026-10-01"))
candles = build_candles_1m(day)
candles.orderBy("candle_ts").show(5, truncate=False)
candles.count()

+-------+-------------------+--------------+--------------+--------------+--------------+-----------+------------------------+-----------+-----------+-----------+
|symbol |candle_ts          |open_price    |close_price   |high_price    |low_price     |volume     |quote_volume            |trade_count|buy_volume |sell_volume|
+-------+-------------------+--------------+--------------+--------------+--------------+-----------+------------------------+-----------+-----------+-----------+
|BTCUSDT|2026-10-01 00:00:00|83623.59000000|83578.01000000|83660.01000000|83578.00000000|28.93519000|2419857.1162231000000000|1346       |14.59132000|14.34387000|
|BTCUSDT|2026-10-01 00:01:00|83578.01000000|83557.25000000|83578.01000000|83550.00000000|12.74545000|1065006.0900790000000000|348        |8.73067000 |4.01478000 |
|BTCUSDT|2026-10-01 00:02:00|83557.25000000|83534.02000000|83557.26000000|83534.01000000|6.99822000 |584685.5653229000000000 |377        |3.24305000 |3.75517000 |
|BTCUSDT|2026-10-01 00

1440

In [7]:
candles.filter((F.col("low_price") > F.col("open_price")) | (
    F.col("high_price") < F.col("close_price"))).count()

0